# Text Search and Spelling Correction System

This notebook preprocesses medical documents, builds an inverted index, and implements k-gram based spelling correction with Soundex phonetic matching.

## Setup

In [1]:
import os
import pandas as pd
from glob import glob
from collections import defaultdict

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import word_tokenize

import csv

nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Load documents

In [2]:
from pathlib import Path
base=Path.cwd()
folder_path=base/'medical_data'
if not folder_path.exists():
    folder_path=base.parent/'Sem2/IR/medical_data'
files=sorted(str(p) for p in folder_path.glob('*'))
print('Folder path', folder_path)
print('Files', files)

def read_file(path):
    ext=path.split('.')[-1]
    if ext=='txt':
        return Path(path).read_text()
    elif ext=='csv':
        import pandas as pd
        df=pd.read_csv(path)
        return ' '.join(df.iloc[0].astype(str))
    elif ext in ('docx','pdf'):
        return Path(path).read_text()
    else:
        return ''

doc_texts=[read_file(p) for p in files]
print('Loaded', len(doc_texts), 'documents')


Folder path /workspace/MTech-AIML/Sem2/IR/medical_data
Files ['/workspace/MTech-AIML/Sem2/IR/medical_data/doc_1.txt', '/workspace/MTech-AIML/Sem2/IR/medical_data/doc_10.pdf', '/workspace/MTech-AIML/Sem2/IR/medical_data/doc_2.txt', '/workspace/MTech-AIML/Sem2/IR/medical_data/doc_3.docx', '/workspace/MTech-AIML/Sem2/IR/medical_data/doc_4.docx', '/workspace/MTech-AIML/Sem2/IR/medical_data/doc_5.pdf', '/workspace/MTech-AIML/Sem2/IR/medical_data/doc_6.pdf', '/workspace/MTech-AIML/Sem2/IR/medical_data/doc_7.csv', '/workspace/MTech-AIML/Sem2/IR/medical_data/doc_8.txt', '/workspace/MTech-AIML/Sem2/IR/medical_data/doc_9.docx']
Loaded 10 documents


## Preprocessing

In [3]:
import string
stop_words=set(stopwords.words('english'))
stemmer=PorterStemmer()
lemmatizer=WordNetLemmatizer()

docs_tokens=[]
counts_pre=[]
counts_stop=[]
counts_norm=[]
counts_stem=[]
for text in doc_texts:
    tokens=word_tokenize(text)
    counts_pre.append(len(tokens))
    tokens=[t.lower() for t in tokens if t.isalpha()]
    counts_norm.append(len(tokens))
    tokens=[t for t in tokens if t not in stop_words]
    counts_stop.append(len(tokens))
    tokens=[lemmatizer.lemmatize(t) for t in tokens]
    counts_stem.append(len(tokens))
    docs_tokens.append(tokens)
print('Tokens before preprocessing:', sum(counts_pre))
print('After normalization:', sum(counts_norm))
print('After stopword removal:', sum(counts_stop))
print('After lemmatization:', sum(counts_stem))


Tokens before preprocessing: 1576
After normalization: 839
After stopword removal: 595
After lemmatization: 595


## Inverted Index

In [4]:
index=defaultdict(set)
for doc_id,tokens in enumerate(docs_tokens):
    for tok in tokens:
        index[tok].add(doc_id)

sorted_index=dict(sorted((term,sorted(list(docs))) for term,docs in index.items()))
for term,docs in list(sorted_index.items())[:10]:
    print(term,':',docs)
print('Total terms:', len(sorted_index))


aberdeen : [7]
academy : [5, 6]
according : [0]
accuracy : [6, 7]
acknowledgement : [4]
adap : [0, 2]
adaptive : [2]
adhesion : [1]
age : [3]
aha : [6]
Total terms: 325


## K-gram Index and Spelling Correction

In [5]:
k=3
kindex=defaultdict(set)
for term in index:
    term_pad=f'${term}$'
    grams=[term_pad[i:i+k] for i in range(len(term_pad)-k+1)]
    for g in grams:
        kindex[g].add(term)

# function to get candidate words for query
from difflib import SequenceMatcher

def candidates(word):
    word_pad=f'${word}$'
    grams=[word_pad[i:i+k] for i in range(len(word_pad)-k+1)]
    cand=set()
    for g in grams:
        cand.update(kindex.get(g, set()))
    # rank by jaccard similarity
    def jaccard(a,b):
        return len(a&b)/len(a|b)
    word_grams=set(grams)
    scored=[(t,jaccard(set(f'${t}$'[i:i+k] for i in range(len(t)+2-k+1)),word_grams)) for t in cand]
    scored.sort(key=lambda x:x[1], reverse=True)
    return [t for t,s in scored[:5]]

print('Candidate suggestions for "diabete":', candidates('diabete'))


Candidate suggestions for "diabete": ['diabetes', 'diagnosis', 'diastolic', 'diagnostic', 'cite']


## Soundex Phonetic Correction

In [6]:
def soundex(word):
    word=word.lower()
    codes={'a':'', 'e':'', 'i':'', 'o':'', 'u':'', 'h':'', 'w':'', 'y':'',
           'b':'1','f':'1','p':'1','v':'1',
           'c':'2','g':'2','j':'2','k':'2','q':'2','s':'2','x':'2','z':'2',
           'd':'3','t':'3',
           'l':'4',
           'm':'5','n':'5',
           'r':'6'}
    first=word[0].upper()
    tail=[codes.get(ch,'') for ch in word[1:]]
    digits=[d for d in tail if d!='']
    res=first+''.join(digits)
    res=res[:4].ljust(4,'0')
    return res

soundex_index=defaultdict(set)
for term in index:
    sound=soundex(term)
    soundex_index[sound].add(term)

word='diabete'
code=soundex(word)
print('Soundex code', code)
print('Phonetic suggestions:', soundex_index.get(code, set()))


Soundex code D130
Phonetic suggestions: {'david'}
